# WGAN-GP Data Augmentation for Rare Emotion Classes

This notebook implements a **Wasserstein GAN with Gradient Penalty (WGAN-GP)** for generating synthetic samples of rare emotion classes to balance the dataset.

**Key Features:**
- Wasserstein loss for stable training
- Gradient penalty for Lipschitz constraint
- Conditional generation (class-specific)
- Mixed precision training for speed

**Rare Classes Target:** DISGUST, SURPRISE(< 15% of max class)

## Section 1: Monta Google Drive
Necessario per leggere il dataset e salvare modelli/output su Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## Section 2: Import e check GPU
Importa TensorFlow/NumPy, controlla versione e disponibilità GPU.

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import h5py, os

print('TF', tf.__version__)
print('GPU available:', len(tf.config.list_physical_devices('GPU')) > 0)

## Section 3: Caricamento dataset HDF5
Carica train/val da `dataset.h5` e unisce i due split per l'analisi.

In [ ]:
DATASET_PATH = '/content/drive/MyDrive/final_scripts/dataset/dataset.h5'
with h5py.File(DATASET_PATH, 'r') as f:
    X_train = np.array(f['X_train'])
    y_train = np.array(f['y_train'])
    X_val = np.array(f['X_val'])
    y_val = np.array(f['y_val'])
    class_names = [c.decode('utf-8') for c in f['class_names']]
X = np.concatenate([X_train, X_val])
y = np.concatenate([y_train, y_val])
print('Shapes', X.shape, y.shape)

## Section 4: Individuazione classi rare
Trova classi con frequenza <15% della classe più numerosa e visualizza la distribuzione.

In [ ]:
counts = np.bincount(y)
max_count = counts.max()
rare_idx = np.where(counts < 0.15 * max_count)[0]
rare_names = [class_names[i] for i in rare_idx]
print('Rare classes:', dict(zip(rare_idx, rare_names)))
for i, n in enumerate(class_names):
    print(f'{n:10s}: {counts[i]}')
plt.bar(class_names, counts)
plt.xticks(rotation=45)
plt.show()

## Section 5: Prepara subset classi rare
Filtra solo le classi rare, normalizza in [-1,1], rimappa etichette a indice locale e applica one-hot.

In [ ]:
mask = np.isin(y, rare_idx)
X_rare = X[mask].astype('float32') / 127.5 - 1.0
y_rare = y[mask]
label_map = {old:new for new, old in enumerate(rare_idx)}
y_remap = np.array([label_map[v] for v in y_rare])
num_classes = len(rare_idx)
y_onehot = tf.keras.utils.to_categorical(y_remap, num_classes)
print('Rare subset', X_rare.shape, y_onehot.shape)

## Section 6: Hyperparameter setup
Definisci dimensione immagine, latente, batch, LR e parametri WGAN-GP (λ_GP, critic_steps).

In [ ]:
IMG_SHAPE = (128,128,3)
NOISE_DIM = 100
BATCH_SIZE = 32
LR = 1e-4
BETA1 = 0.5
LAMBDA_GP = 10
CRITIC_STEPS = 5

## Section 7: Definizione modelli WGAN-GP
Generator: noise+label → immagine. Critic condizionato (senza BatchNorm).

In [ ]:
def build_generator():
    noise = tf.keras.Input(shape=(NOISE_DIM,))
    label = tf.keras.Input(shape=(num_classes,))
    x = tf.keras.layers.Concatenate()([noise, label])
    x = tf.keras.layers.Dense(16*16*256, activation='relu')(x)
    x = tf.keras.layers.Reshape((16,16,256))(x)
    for f in [256,128,64]:
        x = tf.keras.layers.Conv2DTranspose(f, 4, strides=2, padding='same', activation='relu')(x)
        x = tf.keras.layers.BatchNormalization()(x)
    out = tf.keras.layers.Conv2D(3, 3, padding='same', activation='tanh')(x)
    return tf.keras.Model([noise, label], out)

def build_critic():
    img = tf.keras.Input(shape=IMG_SHAPE)
    label = tf.keras.Input(shape=(num_classes,))
    l = tf.keras.layers.Reshape((1,1,num_classes))(label)
    l = tf.keras.layers.UpSampling2D((IMG_SHAPE[0], IMG_SHAPE[1]))(l)
    x = tf.keras.layers.Concatenate()([img, l])
    for f in [64,128,256,512]:
        x = tf.keras.layers.Conv2D(f, 4, strides=2, padding='same')(x)
        x = tf.keras.layers.LeakyReLU(0.2)(x)
        x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Flatten()(x)
    out = tf.keras.layers.Dense(1)(x)
    return tf.keras.Model([img, label], out)

G = build_generator()
C = build_critic()
G.summary(); C.summary()

## Section 8: Funzioni WGAN-GP (loss + gradient penalty)
Implementa il gradient penalty e le loss WGAN (no sigmoid, critic in logit).

In [ ]:
gen_opt = tf.keras.optimizers.Adam(LR, beta_1=BETA1, beta_2=0.9)
crit_opt = tf.keras.optimizers.Adam(LR, beta_1=BETA1, beta_2=0.9)

@tf.function
def gradient_penalty(real_img, fake_img, labels):
    alpha = tf.random.uniform([tf.shape(real_img)[0], 1,1,1], 0.0, 1.0)
    inter = alpha * real_img + (1 - alpha) * fake_img
    with tf.GradientTape() as gp_tape:
        gp_tape.watch(inter)
        pred = C([inter, labels], training=True)
    grads = gp_tape.gradient(pred, inter)
    norm = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=[1,2,3]) + 1e-10)
    return tf.reduce_mean((norm - 1.0) ** 2)

## Section 9: Training loop WGAN-GP
Critic step ripetuto (n=5), poi update generator; logga le loss.

In [ ]:
@tf.function
def train_step(real_img, labels):
    bs = tf.shape(real_img)[0]
    # Update Critic n volte
    for _ in tf.range(CRITIC_STEPS):
        noise = tf.random.normal((bs, NOISE_DIM))
        with tf.GradientTape() as tape:
            fake_img = G([noise, labels], training=True)
            real_out = C([real_img, labels], training=True)
            fake_out = C([fake_img, labels], training=True)
            gp = gradient_penalty(real_img, fake_img, labels)
            c_loss = tf.reduce_mean(fake_out) - tf.reduce_mean(real_out) + LAMBDA_GP * gp
        grads = tape.gradient(c_loss, C.trainable_variables)
        crit_opt.apply_gradients(zip(grads, C.trainable_variables))
    # Update Generator
    noise = tf.random.normal((bs, NOISE_DIM))
    with tf.GradientTape() as tape:
        fake_img = G([noise, labels], training=True)
        fake_out = C([fake_img, labels], training=True)
        g_loss = -tf.reduce_mean(fake_out)
    grads = tape.gradient(g_loss, G.trainable_variables)
    gen_opt.apply_gradients(zip(grads, G.trainable_variables))
    return c_loss, g_loss

def train(epochs=150):
    ds = tf.data.Dataset.from_tensor_slices((X_rare, y_onehot)).shuffle(len(X_rare)).batch(BATCH_SIZE)
    c_hist, g_hist = [], []
    for epoch in range(epochs):
        c_loss_epoch = g_loss_epoch = 0.0; steps = 0
        for real_img, labels in ds:
            c_loss, g_loss = train_step(real_img, labels)
            c_loss_epoch += c_loss; g_loss_epoch += g_loss; steps += 1
        c_hist.append(float(c_loss_epoch/steps)); g_hist.append(float(g_loss_epoch/steps))
        if (epoch+1) % 20 == 0:
            print(f'Epoch {epoch+1}/{epochs} - Critic: {c_hist[-1]:.3f} - Gen: {g_hist[-1]:.3f}')
    return c_hist, g_hist

c_losses, g_losses = train(epochs=120)
plt.plot(c_losses, label='Critic'); plt.plot(g_losses, label='Gen'); plt.legend(); plt.show()

## Section 10: Generazione sintetici
Per ogni classe rara genera campioni (≈25% del max), denormalizza a uint8.

In [ ]:
target_per_class = int(max_count * 0.25)
synth_images = []
synth_labels = []
for cid in range(num_classes):
    noise = tf.random.normal((target_per_class, NOISE_DIM))
    labels = np.zeros((target_per_class, num_classes), dtype=np.float32)
    labels[:, cid] = 1
    imgs = G.predict([noise, labels], verbose=0)
    synth_images.append(imgs)
    synth_labels += [cid]*target_per_class
synth_images = np.concatenate(synth_images)
synth_images_uint8 = ((synth_images + 1)/2 * 255).astype(np.uint8)
synth_labels = np.array(synth_labels)
print('Synthetic set', synth_images_uint8.shape, synth_labels.shape)

## Section 11: Anteprima sintetici
Visualizza una griglia di esempi generati per controllo qualitativo.

In [ ]:
fig, ax = plt.subplots(2,5, figsize=(10,4))
for i in range(10):
    r, c = divmod(i,5)
    ax[r,c].imshow(synth_images_uint8[i])
    ax[r,c].axis('off')
plt.show()

## Section 12: Salvataggio modelli e dataset sintetico
Salva generator/critic e i file `.npy` (immagini, etichette, indici classi rare) su Drive.

In [ ]:
OUT_DIR = '/content/drive/MyDrive/WGAN_GP_augmentation'
os.makedirs(OUT_DIR, exist_ok=True)
G.save(os.path.join(OUT_DIR, 'generator_wgan_gp.h5'))
C.save(os.path.join(OUT_DIR, 'critic_wgan_gp.h5'))
np.save(os.path.join(OUT_DIR, 'synthetic_images.npy'), synth_images_uint8)
np.save(os.path.join(OUT_DIR, 'synthetic_labels.npy'), synth_labels)
np.save(os.path.join(OUT_DIR, 'rare_class_indices.npy'), rare_idx)
print('Saved to', OUT_DIR)